In [ ]:
import os
from tqdm.auto import tqdm
import pandas as pd

import avoddiag as ag

# Load Datasets

In [ ]:
image_dataset_A = ag.image.data.Dataset(
    root_folder_path = 'output/TestDataset_01_Generated_Images'
)

In [ ]:
image_dataset_B = ag.image.data.Dataset(
    root_folder_path = 'output/TestDataset_02_Edited_Images'
)

# Merge based on filtered metadata

In [ ]:
image_dataset_A.metadata['attributes_filtered:gemini-2.5-flash']

In [ ]:
image_dataset_B.metadata['attributes_filtered:gemini-2.5-flash']

In [ ]:
# Create a new emtpy dataset that will hold the merged data
image_dataset_merged = ag.image.data.Dataset(
    root_folder_path = 'output/TestDataset_03_Generated+Edited_Images'
)
image_dataset_merged.make_dirs()

In [ ]:
# Create image symlinks
file_names_mapping = []
for idx, image_dataset in enumerate([image_dataset_A, image_dataset_B]):
    print(f'Processing dataset {idx+1}')
    
    for image_file_name in tqdm(image_dataset.metadata['attributes_filtered:gemini-2.5-flash']['image_file_name'].to_list(), desc='Create symlinks'):
        image_data = image_dataset[image_file_name]

        src_image_file_path = image_data['image_file_path']

        dst_image_file_name = f'{idx:0>2d}_{image_file_name}'
        dst_image_file_path = os.path.join(image_dataset_merged.images_folder_path, dst_image_file_name)
        assert not os.path.isfile(dst_image_file_path), f'Destination image file already exists: {dst_image_file_path}'

        os.symlink(
            os.path.relpath(
                src_image_file_path,
                os.path.dirname(dst_image_file_path)
            ),
            dst_image_file_path
        )

        file_names_mapping.append((idx, image_file_name, dst_image_file_name))

file_names_mapping_df = pd.DataFrame(file_names_mapping, columns=['dataset_idx', 'src_image_file_name', 'dst_image_file_name']) 
file_names_mapping_df

In [ ]:
# Merge metadata
sorted(image_dataset_A.metadata.keys())

In [ ]:
sorted(image_dataset_B.metadata.keys())

In [ ]:
if 'attribute:vehicle_presence' not in image_dataset_A.metadata['attributes_input']:
    image_dataset_A.metadata['attributes_input']['attribute:vehicle_presence'] = image_dataset_A.metadata['attributes_input']['attribute:vehicle_count'] > 0
    
image_dataset_A.metadata['attributes_input']

In [ ]:
image_dataset_B.metadata['attributes_input']

In [ ]:
fns_A = set(image_dataset_A.metadata['attributes_filtered:gemini-2.5-flash']['image_file_name'].to_list())
fns_B = set(image_dataset_B.metadata['attributes_filtered:gemini-2.5-flash']['image_file_name'].to_list())

mask_A = image_dataset_A.metadata['attributes_input']['image_file_name'].apply(fns_A.__contains__)
mask_B = image_dataset_B.metadata['attributes_input']['image_file_name'].apply(fns_B.__contains__)

file_names_mapping_A = file_names_mapping_df[file_names_mapping_df['dataset_idx'] == 0].set_index('src_image_file_name')[['dst_image_file_name']].to_dict()['dst_image_file_name']
file_names_mapping_B = file_names_mapping_df[file_names_mapping_df['dataset_idx'] == 1].set_index('src_image_file_name')[['dst_image_file_name']].to_dict()['dst_image_file_name']

df_A = image_dataset_A.metadata['attributes_input'][mask_A][
    [
        'image_file_name',
        'attribute:scene_type',
        'attribute:season',
        'attribute:weather',
        'attribute:vehicle_presence',
        'prompt',
        # 'is_edited'
    ]
].copy()
df_A['is_edited'] = False
df_A['image_file_name'] = df_A['image_file_name'].apply(lambda x: file_names_mapping_A[x])

df_B = image_dataset_B.metadata['attributes_input'][mask_B][
    [
        'image_file_name',
        'attribute:scene_type',
        'attribute:season',
        'attribute:weather',
        'attribute:vehicle_presence',
        # 'prompt_editing:gemini-2.5-flash-lite'
    ]
].rename(columns={'prompt_editing:gemini-2.5-flash-lite': 'prompt'})
df_B['is_edited'] = True
# df_B['is_edited'] = False
df_B['image_file_name'] = df_B['image_file_name'].apply(lambda x: file_names_mapping_B[x])

attributes_input = pd.concat(
    [
        df_A,
        df_B
    ]
).reset_index(drop=True)

image_dataset_merged.metadata['attributes_input'] = attributes_input
attributes_input

In [ ]:
attributes_filtered_A = image_dataset_A.metadata['attributes_filtered:gemini-2.5-flash'][
    [
        'image_file_name',
        'in:scene',
        'in:season',
        'in:weather',
        'in:vehicle_presence',
        'gen:scene',
        'gen:scene_concepts',
        'gen:scene_confirmation',
        'gen:scene_adjusted',
        'gen:season',
        'gen:season_confirmation',
        'gen:season_adjusted',
        'gen:weather',
        'gen:weather_confirmation',
        'gen:weather_adjusted',
        'gen:vehicle_presence_confirmation',
        'gen:vehicle_presence',
    ]
]
attributes_filtered_A['image_file_name'] = attributes_filtered_A['image_file_name'].apply(lambda x: file_names_mapping_A[x])

attributes_filtered_B = image_dataset_B.metadata['attributes_filtered:gemini-2.5-flash'][
    [
        'image_file_name',
        'in:scene',
        'in:season',
        'in:weather',
        'in:vehicle_presence',
        'gen:scene',
        'gen:scene_concepts',
        'gen:scene_confirmation',
        'gen:scene_adjusted',
        'gen:season',
        'gen:season_confirmation',
        'gen:season_adjusted',
        'gen:weather',
        'gen:weather_confirmation',
        'gen:weather_adjusted',
        'gen:vehicle_presence_confirmation',
        'gen:vehicle_presence',
    ]
]
attributes_filtered_B['image_file_name'] = attributes_filtered_B['image_file_name'].apply(lambda x: file_names_mapping_B[x])

attributes_filtered = pd.concat([attributes_filtered_A, attributes_filtered_B]).reset_index(drop=True)

image_dataset_merged.metadata['attributes_filtered:gemini-2.5-flash'] = attributes_filtered
attributes_filtered

In [ ]:
# SAVE METADATA
image_dataset_merged.save_metadata(keys_to_overwrite=['attributes_filtered:gemini-2.5-flash', 'attributes_input'])

In [ ]:
image_dataset_merged.metadata.keys()

In [ ]:
image_dataset_merged.metadata['attributes_input']

In [ ]:
image_dataset_merged.metadata['attributes_filtered:gemini-2.5-flash']

In [ ]:
image_dataset_merged.image_file_names